# 📈 Forecast de Ventas con Prophet
> **Caso de negocio:** Crisol · Planificación de demanda e-commerce
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Crisol opera una tienda online de libros, manga y artículos de colección. El equipo de operaciones planifica inventario y el equipo de marketing planifica campañas **a ojo**, sin un pronóstico formal de ventas diarias.

**Objetivo:** Pronosticar las ventas diarias para los próximos 30 días, capturando la tendencia de crecimiento, la estacionalidad semanal (fines de semana) y la estacionalidad anual (temporada escolar / campaña navideña).

**KPIs de éxito:**
- MAPE <= 10% en el set de prueba (últimos 30 días)
- El forecast debe capturar el patrón semanal (más ventas en fin de semana)
- Intervalo de confianza del 95% que cubra la mayoría de los valores reales

**Algoritmo:** Prophet (Meta) — modelo aditivo de series de tiempo con tendencia + estacionalidades

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q prophet plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Forecasting de series de tiempo univariadas',
    'target': 'ventas (unidades vendidas por día)',
    'horizonte_forecast': '30 días',
    'componentes_esperados': [
        'tendencia      → crecimiento estructural del e-commerce',
        'estacionalidad semanal → picos en fin de semana',
        'estacionalidad anual   → temporada escolar (marzo) y campaña navideña (diciembre)',
    ],
    'criterios_aceptacion': {
        'MAPE': '<= 10%',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos: 2 años de ventas diarias
# En producción: reemplazar con agregación diaria de pedidos (BigQuery)
N_DAYS = 730
dates = pd.date_range('2023-01-01', periods=N_DAYS, freq='D')

t = np.arange(N_DAYS)
trend = 800 + 0.6 * t

dayofweek = dates.dayofweek.to_numpy()
dayofyear = dates.dayofyear.to_numpy()
is_weekend = np.isin(dayofweek, [5, 6]).astype(float)

weekly = 50 * np.sin(2 * np.pi * dayofweek / 7) + 90 * is_weekend
yearly = 180 * np.sin(2 * np.pi * (dayofyear - 60) / 365.25)
noise = np.random.normal(0, 45, N_DAYS)

ventas = np.round(trend + weekly + yearly + noise).clip(min=100).astype(int)

df = pd.DataFrame({'fecha': dates, 'ventas': ventas})

print(f'Dataset: {df.shape}')
print(f'Rango de fechas: {df["fecha"].min().date()} a {df["fecha"].max().date()}')
df.describe().round(1)

In [ ]:
# Serie de tiempo completa
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['fecha'], y=df['ventas'], mode='lines',
                          line=dict(color='#1a4c8c', width=1.2), name='Ventas'))
fig.update_layout(title='Ventas diarias — Crisol e-commerce (2 años)',
                  height=380, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0'), yaxis=dict(gridcolor='#f0f0f0', title='Unidades'))
fig.show()

In [ ]:
# Patrón semanal: ventas promedio por día de la semana
nombres_dia = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
ventas_por_dia = df.groupby(df['fecha'].dt.dayofweek)['ventas'].mean()

fig = go.Figure(go.Bar(x=nombres_dia, y=ventas_por_dia.values,
                        marker_color=['#1a4c8c']*5 + ['#0d9488']*2,
                        text=ventas_por_dia.round(0).astype(int), textposition='outside'))
fig.update_layout(title='Ventas promedio por día de la semana',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#f0f0f0', title='Unidades'))
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
# Prophet requiere columnas 'ds' (fecha) y 'y' (valor)
data = df.rename(columns={'fecha': 'ds', 'ventas': 'y'}).copy()

N_TEST = 30
train = data.iloc[:-N_TEST].reset_index(drop=True)
test = data.iloc[-N_TEST:].reset_index(drop=True)

print(f'Train: {len(train)} días ({train["ds"].min().date()} a {train["ds"].max().date()})')
print(f'Test:  {len(test)} días ({test["ds"].min().date()} a {test["ds"].max().date()})')

## 4️⃣ Fase 4 — Modeling

In [ ]:
SIM_DAYS = 30

model = Prophet(
    changepoint_prior_scale=0.05,
    seasonality_mode='additive',
    yearly_seasonality=True,
    weekly_seasonality=True,
)
model.fit(train)

future = model.make_future_dataframe(periods=N_TEST + SIM_DAYS)
forecast = model.predict(future)

print(f'Forecast generado: {len(forecast)} días (incluye {N_TEST} días de test + {SIM_DAYS} días futuros)')
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(3)

## 5️⃣ Fase 5 — Evaluation

In [ ]:
fc_test = forecast.set_index('ds').loc[test['ds']]
y_true = test['y'].values
y_pred = fc_test['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100)

metrics = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
criterios = {'MAPE': 10.0}

print('=== MÉTRICAS EN TEST SET (últimos 30 días) ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v <= umbral else f'❌ NECESITA MEJORA (umbral <= {umbral}%)'
    suffix = '%' if k == 'MAPE' else ''
    print(f'  {k:6s}: {v:.2f}{suffix}  {estado}')

In [ ]:
# Forecast vs real, con intervalo de confianza del 95%
fig = go.Figure()

fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines',
                          line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines',
                          line=dict(width=0), fill='tonexty', fillcolor='rgba(26,76,140,0.15)',
                          name='Intervalo 95%', hoverinfo='skip'))
fig.add_trace(go.Scatter(x=data['ds'], y=data['y'], mode='markers',
                          marker=dict(color='#6b7280', size=4, opacity=0.5), name='Real'))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines',
                          line=dict(color='#1a4c8c', width=2), name='Forecast'))
fig.add_vline(x=test['ds'].iloc[0], line_dash='dash', line_color='#c0392b')

fig.update_layout(title='Forecast vs Real — Prophet',
                  height=420, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', range=[test['ds'].iloc[0] - pd.Timedelta(days=60), forecast['ds'].max()]),
                  yaxis=dict(gridcolor='#f0f0f0', title='Unidades'))
fig.show()

In [ ]:
# Componentes del modelo: tendencia, estacionalidad semanal y anual
trend_fig = go.Scatter(x=forecast['ds'], y=forecast['trend'], mode='lines',
                        line=dict(color='#1a4c8c', width=2))

wk = forecast[['ds', 'weekly']].copy()
wk['dow'] = wk['ds'].dt.dayofweek
wk = wk.drop_duplicates('dow').sort_values('dow')
weekly_fig = go.Bar(x=nombres_dia, y=wk['weekly'].values, marker_color='#0d9488')

yr = forecast[['ds', 'yearly']].copy()
yr['doy'] = yr['ds'].dt.dayofyear
yr = yr.drop_duplicates('doy').sort_values('doy')
yearly_fig = go.Scatter(x=yr['ds'], y=yr['yearly'], mode='lines',
                         line=dict(color='#7c3aed', width=2))

fig = make_subplots(rows=3, cols=1, subplot_titles=['Tendencia', 'Estacionalidad semanal', 'Estacionalidad anual'])
fig.add_trace(trend_fig, row=1, col=1)
fig.add_trace(weekly_fig, row=2, col=1)
fig.add_trace(yearly_fig, row=3, col=1)
fig.update_layout(height=700, showlegend=False, plot_bgcolor='white', paper_bgcolor='white',
                  title='Componentes del modelo Prophet')
fig.update_xaxes(gridcolor='#f0f0f0')
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Forecast operativo: próximos 30 días (después del último día histórico)
ultimo_dia_real = data['ds'].max()
forecast_futuro = forecast[forecast['ds'] > ultimo_dia_real].head(30).copy()
forecast_futuro = forecast_futuro[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].round(0)
forecast_futuro.columns = ['fecha', 'ventas_estimadas', 'limite_inferior', 'limite_superior']

print(f'Forecast operativo: {len(forecast_futuro)} días desde {forecast_futuro["fecha"].min().date()}')
forecast_futuro.head(10)

In [ ]:
# Guardar outputs
forecast_futuro.to_csv('forecast_ventas_crisol.csv', index=False)
print('Archivo forecast_ventas_crisol.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'mape_test': mape}, 'modelo_prophet_crisol.pkl')
print('Modelo modelo_prophet_crisol.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
ventas_promedio_futuro = forecast_futuro['ventas_estimadas'].mean()
ventas_promedio_hist = data['y'].tail(30).mean()
crecimiento = (ventas_promedio_futuro / ventas_promedio_hist - 1) * 100

print('=== RESUMEN EJECUTIVO ===')
print(f'Días históricos analizados:          {len(data):,}')
print(f'MAPE en test (últimos 30 días):      {mape:.2f}%')
print(f'Ventas promedio próximos 30 días:    {ventas_promedio_futuro:,.0f} u/día')
print(f'Variación vs. últimos 30 días reales: {crecimiento:+.1f}%')
print(f'\nArquitectura de producción:')
print('  Pedidos → BigQuery → reentrenamiento semanal del modelo → forecast en dashboard de Operaciones/Marketing')